In [ ]:
def chatbot(state: OrderState) -> OrderState:
    """The basic chatbot node that invokes the LLM."""
    message_history = [BARISTABOT_SYSINT] + state["messages"]
    return {"messages": [llm.invoke(message_history)]}


# Set up the initial graph based on our state definition
graph_builder = StateGraph(OrderState)

# Add the chatbot function
graph_builder.add_node("chatbot", chatbot)

# Define the chatbot node as the app entrypoint
graph_builder.add_edge(START, "chatbot")

# We'll compile just to show the structure for now
chat_graph = graph_builder.compile()

print("Basic graph created. Structure:")
print(chat_graph.get_graph())

In [ ]:
class OrderState(TypedDict):
    """State representing the customer's order conversation."""
    
    # The chat conversation. This preserves the conversation history
    # between nodes. The `add_messages` annotation indicates to LangGraph
    # that state is updated by appending returned messages, not replacing them.
    messages: Annotated[list, add_messages]
    
    # The customer's in-progress order.
    order: list[str]
    
    # Flag indicating that the order is placed and completed.
    finished: bool


# The system instruction defines how the chatbot is expected to behave
BARISTABOT_SYSINT = (
    "system",
    "You are a BaristaBot, an interactive cafe ordering system. A human will talk to you about the "
    "available products you have and you will answer any questions about menu items (and only about "
    "menu items - no off-topic discussion, but you can chat about the products and their history). "
    "The customer will place an order for 1 or more items from the menu, which you will structure "
    "and send to the ordering system after confirming the order with the human. "
    "\n\n"
    "Add items to the customer's order with add_to_order, and reset the order with clear_order. "
    "To see the contents of the order so far, call get_order (this is shown to you, not the user) "
    "Always confirm_order with the user (double-check) before calling place_order. Calling confirm_order will "
    "display the order items to the user and returns their response to seeing the list. Their response may contain modifications. "
    "Always verify and respond with drink and modifier names from the MENU before adding them to the order. "
    "If you are unsure a drink or modifier matches those on the MENU, ask a question to clarify or redirect. "
    "You only have the modifiers listed on the menu. "
    "Once the customer has finished ordering items, call confirm_order to ensure it is correct then make "
    "any necessary updates and then call place_order. Once place_order has returned, thank the user and "
    "say goodbye!",
)

# This is the message with which the system opens the conversation
WELCOME_MSG = "Welcome to the BaristaBot cafe. Type 'q' to quit. How may I serve you today?"

print("State and system instructions defined!")

In [ ]:
# API Key Setup
# If you're running locally, set GOOGLE_API_KEY environment variable with your API key
# Get your API key from: https://ai.google.dev/
# If running in Kaggle, the API key should be set from the secrets

if not os.getenv("GOOGLE_API_KEY"):
    # Try to get from Kaggle secrets
    try:
        from kaggle_secrets import UserSecretsClient
        GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
        os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
        print("API key loaded from Kaggle secrets")
    except:
        print("Warning: GOOGLE_API_KEY not set. Please set it in your environment.")
else:
    print("API key already set in environment")

# Initialize the Gemini LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-latest")
print("LLM initialized successfully!")

In [ ]:
import os
from typing import Annotated, Literal
from typing_extensions import TypedDict
from collections.abc import Iterable
from random import randint
from pprint import pprint

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.messages.tool import ToolMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, InjectedState
from IPython.display import Image

print("All imports successful!")

In [ ]:
%pip install -qU "langgraph==1.0.5" "langchain-google-genai==4.1.2" "google-genai==1.56.0" "langchain-core" "pillow"
print("Dependency installation completed successfully!")

# BaristaBot: Building a Cafe Ordering System with LangGraph and Gemini API

This notebook demonstrates how to create a stateful application using LangGraph that integrates with the Gemini API to build an interactive cafe ordering system called **BaristaBot**.

## Learning Objectives
- Create stateful applications using LangGraph
- Integrate Gemini API (via LangChain) into LangGraph applications
- Define and manipulate state using TypedDict
- Simulate dynamic, tool-augmented behavior with menus and ordering
- Model conditional transitions and loops for user interaction
- Handle tool calls using LangGraph's ToolNode mechanism

## What We'll Build
A conversational cafe ordering system (BaristaBot) that:
- Takes coffee/tea orders using natural language
- Offers a real-time menu via tools
- Confirms and modifies orders
- Loops through conversation until an order is placed
- Handles tool calls using LangGraph's ToolNode